In [ ]:
# Day 28 - Custom YOLO Training (v1: yolov8s, no augmentation)



!pip install ultralytics roboflow -q

from roboflow import Roboflow
from ultralytics import YOLO
import os
import pandas as pd
import random
import shutil

from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/Day29_Models"
os.makedirs(drive_folder, exist_ok=True)

# Step 1: Download dataset

api_key = userdata.get("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sdp-lfigk").project("ppe-detection-ozhfb")
version = project.version(14)
dataset = version.download("yolov8")

print(f"Dataset downloaded at: {dataset.location}")

# Step 2: Explore dataset

for split in ["train", "valid", "test"]:
    img_path = os.path.join(dataset.location, split, "images")
    if os.path.exists(img_path):
        print(f"{split}: {len(os.listdir(img_path))} images")

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path, "r") as f:
    print(f.read())

# Step 3: Train the model (v1 baseline, no augmentation)

model = YOLO("yolov8s.pt")

results = model.train(
    data=yaml_path,
    epochs=60,
    batch=16,
    imgsz=640,
    patience=15,
    project="results",
    name="ppe_detection_run",
)

# Step 4: Monitor training (dynamic path)


run_dir = results.save_dir
results_csv_path = str(run_dir / "results.csv")
if os.path.exists(results_csv_path):
    log_df = pd.read_csv(results_csv_path)
    log_df.columns = log_df.columns.str.strip()
    print("Last 5 epochs summary:")
    print(log_df[["epoch", "train/box_loss", "train/cls_loss",
                   "metrics/precision(B)", "metrics/recall(B)",
                   "metrics/mAP50(B)"]].tail(5))

# Step 5: Evaluate + target check


metrics = model.val()
map50 = metrics.box.map50
map50_95 = metrics.box.map

print(f"mAP@50: {map50:.4f}")
print(f"mAP@50-95: {map50_95:.4f}")

TARGET_MAP50 = 0.80
if map50 >= TARGET_MAP50:
    print(f"Target achieved! mAP@50 ({map50:.2%}) meets the {TARGET_MAP50:.0%} goal.")
else:
    print("Target NOT met.")

print("\nPer-class mAP@50:")
for i, class_name in enumerate(model.names.values()):
    print(f"  {class_name}: {metrics.box.ap50[i]:.4f}")

# Step 6: Inference on 10 test images (dynamic path)



best_model_path = str(run_dir / "weights" / "best.pt")
print(f"Loading model from: {best_model_path}")
trained_model = YOLO(best_model_path)

test_images_dir = os.path.join(dataset.location, "test", "images")
if not os.path.exists(test_images_dir):
    print("No test folder found — using valid/images instead.")
    test_images_dir = os.path.join(dataset.location, "valid", "images")

all_test_images = os.listdir(test_images_dir)
sample_size = min(10, len(all_test_images))
selected_images = random.sample(all_test_images, sample_size)
selected_paths = [os.path.join(test_images_dir, img) for img in selected_images]

print(f"Running inference on {len(selected_paths)} test images.")

predictions = trained_model.predict(
    source=selected_paths,
    save=True,
    conf=0.4,
    project="Prediction Results",
    name="test_inference",
)

for i, pred in enumerate(predictions):
    print(f"Image {i+1}: {len(pred.boxes)} objects detected")


# Step 7: Copy sample test images



os.makedirs("Sample Test Images", exist_ok=True)
for img_path in selected_paths:
    shutil.copy(img_path, "Sample Test Images/")

print("Sample test images copied.")


# Step 8: SAVE PERMANENTLY TO GOOGLE DRIVE



drive_best_path = os.path.join(drive_folder, "best.pt")
shutil.copy(best_model_path, drive_best_path)
print(f"Model permanently saved to Google Drive: {drive_best_path}")

# Also copy the results.csv and full run folder for backup (optional but useful)
drive_run_backup = os.path.join(drive_folder, "ppe_detection_run")
if not os.path.exists(drive_run_backup):
    shutil.copytree(str(run_dir), drive_run_backup)
    print(f"Full run folder backed up to: {drive_run_backup}")


# Step 9: Also download locally



from google.colab import files
files.download(best_model_path)